# Unit Test Suite: Environment & Component Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ltellado/LAQuer-Audit/blob/main/notebooks/tests/laquer-audit-unit-tests.ipynb)

This notebook executes isolated unit tests for the core modules supporting the project **"Reproducing and Extending LAQuer for Post-Hoc Attribution"**.

### Objectives
1. **Environment Sanity**: Validate platform detection, directory structure, baseline submodule integrity (`eranhirs/LAQuer` pinned at commit `a4affb2`), credential retrieval, and PyTorch hardware acceleration (CUDA/MPS/CPU).
2. **Component Isolation**: Exercise each pipeline stage individually to verify API contracts, string normalization, and verification invariants before running full-scale experimental batches.

---

In [5]:
# -------------------------------------------------------------------------
# Environment Setup: Anchor working directory & sync latest code
# -------------------------------------------------------------------------
import os, sys

if "google.colab" in sys.modules:
    repo_dir = "/content/LAQuer-Audit"
    if not os.path.exists(repo_dir):
        !git clone https://github.com/ltellado/LAQuer-Audit.git {repo_dir}
        %cd {repo_dir}
        !bash setup.sh
    else:
        %cd {repo_dir}
        # Discard any local modifications and pull latest remote commits
        !git reset --hard HEAD
        !git pull origin main
else:
    # Local execution (VS Code): traverse up until repo root (with setup.sh)
    cur = os.getcwd()
    while cur != "/" and not os.path.exists(os.path.join(cur, "setup.sh")):
        cur = os.path.dirname(cur)
    if os.path.exists(os.path.join(cur, "setup.sh")):
        %cd {cur}

print("Active Working Directory:", os.getcwd())

/Users/leo/Workspaces/MSAI/LAQuer-Audit
Active Working Directory: /Users/leo/Workspaces/MSAI/LAQuer-Audit


## Test 1: `LAQuerAudit` Environment Bootstrap (`src/bootstrap.py`)

Validates the static bootstrapping utility:
- **Platform Detection**: Distinguishes between local runtimes (VS Code / CLI) and Google Colab.
- **Repository Root Resolution**: Dynamically discovers root paths and registers `src/` on `sys.path`.
- **Baseline Submodule Integrity**: Confirms the pinned `LAQuer` submodule exists.
- **Credential Access**: Loads `OPENAI_API_KEY` from the environment or Colab Secrets (`userdata`).
- **Device Arbitration**: Resolves the optimal compute device (`cuda`, `mps`, or `cpu`).

**Pass Criteria**: Outputs `✓ Unit test passed for LAQuerAudit!` with no unhandled exceptions.

In [2]:
# Test LAQuerAudit
%run src/bootstrap.py

=== [Unit Test] LAQuerAudit ===
Is Google Colab:  False
Repository Root:  /Users/leo/Workspaces/MSAI/LAQuer-Audit
PyTorch Device:   mps
API Key Detected: True

✓ Unit test passed for LAQuerAudit!



## Test 2: `LAQuerLocalizer` LLM In-Context Localizer (`src/localizer.py`)

Tests the core LLM span-extraction module (Hirsch et al., ACL 2025):
- **Prompt Construction**: Compiles the official instruction prompt (`ins_prompt`) and the 3 LFQA few-shot exemplars with target source documents and claim.
- **Deterministic Inference**: Dispatches to OpenAI chat completion with `temperature=0`.
- **Boundary Stripping**: Normalizes trailing boundary punctuation (`clean_span_boundary`) that commercial chat models append to mid-sentence spans.
- **Extractive Invariant Verification**: Asserts that every extracted span matches at least one source document verbatim as an exact substring.

**Pass Criteria**: Outputs parsed spans, reports `All Verbatim Match: True`, and concludes with `✓ Unit test passed for LAQuerLocalizer!`.

In [3]:
# Test LAQuerLocalizer
%run src/localizer.py

=== [Unit Test] LAQuerLocalizer ===
Executing localization with gpt-4o-mini...
Raw Output:         Reparations compensate victims for abuses suffered during periods of conflict.
Parsed Spans:       ['Reparations compensate victims for abuses suffered during periods of conflict.']
Verbatim Flags:     [True]
All Verbatim Match: True
Prompt Tokens:      577
Completion Tokens:  13

✓ Unit test passed for LAQuerLocalizer!



## Test 3: `LAQuerVerifier` NLI Cross-Encoder (`src/verifier.py`)

Tests the cross-encoder verification module using DeBERTa-v3 (`cross-encoder/nli-deberta-v3-base`):
- **Hardware Acceleration**: Confirms automatic device routing via `LAQuerAudit.resolve_device()` (`cuda` in Colab, `mps` on Apple Silicon, or `cpu`).
- **NLI Classification**: Evaluates pairwise classification probabilities across three distinct relationships:
  1. *Entailment* (supporting attribution span entails claim)
  2. *Contradiction* (conflicting factual claims)
  3. *Neutral* (unrelated or unsupported premises)
- **Batch Processing**: Exercises `batch_verify` for accelerated multi-premise inference.

**Pass Criteria**: Correctly predicts class labels for all three test cases and prints `✓ Unit test passed for LAQuerVerifier!`.

In [4]:
# Test LAQuerVerifier
%run src/verifier.py

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== [Unit Test] LAQuerVerifier ===
Initializing LAQuerVerifier on device: mps...


W1127 18:14:01.527000 49922 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 7480.31it/s]



Case:             Positive Entailment
Premise:          Reparations compensate victims for abuses suffered during periods of conflict.
Hypothesis:       Reparations compensate victims for abuses suffered.
Predicted Label:  entailment
Entailment Prob:  0.9931
Is Entailed:      True

Case:             Contradiction
Premise:          Reparations compensate victims for abuses suffered during periods of conflict.
Hypothesis:       Reparations never provide compensation to victims of conflict.
Predicted Label:  contradiction
Entailment Prob:  0.0000
Is Entailed:      False

Case:             Neutral / Unrelated
Premise:          Truth commissions investigate past human rights violations to promote societal reconciliation.
Hypothesis:       Economic sanctions were immediately imposed by the council.
Predicted Label:  neutral
Entailment Prob:  0.0000
Is Entailed:      False

✓ Unit test passed for LAQuerVerifier!



## Test 4: `LAQuerDecomposer` FActScore Atomic Decomposition (`src/decomposer.py`)

Tests the highlight synthesis module following Min et al. (EMNLP 2023) and Choi et al. (TACL 2021):
- **Atomic Fact Decomposition**: Breaks down complex compound sentences into single-assertion propositions.
- **Molecular Decontextualization**: Resolves ambiguous pronouns (`they`, `he`, `it`) against the paragraph context so claims stand alone for NLI verification.

**Pass Criteria**: Outputs $\ge 2$ atomic facts, resolves antecedent pronouns, and prints `✓ Unit test passed for LAQuerDecomposer!`.

In [6]:
# Test LAQuerDecomposer
%run src/decomposer.py

=== [Unit Test] LAQuerDecomposer ===
Input Sentence:
  Reparations are not inherently a form of punishment; rather, they are measures intended to compensate victims for abuses suffered, acknowledging their suffering.

Decomposing into atomic facts...
Extracted Facts (4):
  1. Reparations are not inherently a form of punishment.
  2. Reparations are measures intended to compensate victims.
  3. Reparations compensate victims for abuses suffered.
  4. Reparations acknowledge the suffering of victims.

Testing decontextualization...
  Context:     The Truth and Reconciliation Commission introduced reparations. They compensate victims for abuses suffered.
  Vague Claim: They compensate victims for abuses suffered.
  Resolved:    The Truth and Reconciliation Commission introduced reparations to compensate victims for abuses suffered.

✓ Unit test passed for LAQuerDecomposer!

